In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
QWEN35_TRANSFORMERS_REVISION = "b70d02fc724d04c916832ca4ead03ff05e8fb1ee"
qwen35_probe = subprocess.run(
    [sys.executable, "-c", "from transformers import AutoModelForMultimodalLM"],
    capture_output=True,
)
if qwen35_probe.returncode != 0:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"transformers @ git+https://github.com/huggingface/transformers.git@{QWEN35_TRANSFORMERS_REVISION}",
        "torchvision", "pillow",
    ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 05. Intervention Mode Comparison

질문: 같은 feature에 대해 제거, 억제, 증폭, projection 제거가 서로 다른 효과를 보이는가?

관점:
- `remove_activation`: SAE 활성값만큼 decoder direction 제거
- `projection_remove`: 현재 residual에서 decoder 방향 성분 자체를 제거
- `subtract_unit`: feature 방향을 단위 벡터 크기로 억제
- `add_activation`/`add_unit`: 반대로 feature를 강화

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_ANALYSIS_PROFILE_KEY,
    LureCase,
    answer_logprob_margin,
    answer_variant_rows,
    bat_ball_answer_variants,
    bat_ball_paraphrases,
    candidate_feature_rows,
    case_transfer_rows,
    coefficient_sweep_for_handle,
    control_delta_bypass_rows,
    crt_transfer_cases,
    decoder_cosine_rows,
    default_sae_device,
    dtype_from_name,
    feature_handle_from_result,
    get_qwen35_analysis_profile,
    instruct_lure_case,
    intervention_mode_rows,
    layer_feature_search_rows,
    load_or_discover_handle_and_sae,
    load_qwen_language_model,
    load_qwen_scope_sae,
    prompt_token_window_rows,
    rank_lure_feature_effects,
    recommended_dtype_name,
    sae_decoder_direction,
    save_feature_handle,
    semantic_lure_cases,
    token_position_sweep_rows,
)

ANALYSIS_PROFILE_KEY = DEFAULT_ANALYSIS_PROFILE_KEY  # 2b, 9b, 27b, 35b-a3b
PROFILE = get_qwen35_analysis_profile(ANALYSIS_PROFILE_KEY)
MODEL_ID = PROFILE.analysis_model_id
SAE_REPO_ID = PROFILE.sae_repo_id
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE
HANDLE_CACHE = root / "outputs" / "candidates" / f"bat_ball_top_feature_answer_instruction_{PROFILE.key}.json"

lm = load_qwen_language_model(MODEL_ID, device_map="auto", dtype=DTYPE, dispatch=True)
print({"profile": PROFILE.key, "model": MODEL_ID, "sae_repo": SAE_REPO_ID, "dtype": DTYPE, "sae_device": SAE_DEVICE})


In [ ]:
CASE = instruct_lure_case(BAT_BALL_CASE)
REFRESH_FEATURE = False

handle, sae, discovery_rows, loaded_from_cache = load_or_discover_handle_and_sae(
    lm,
    CASE,
    repo_id=SAE_REPO_ID,
    cache_path=HANDLE_CACHE,
    default_layer=14,
    sae_device=SAE_DEVICE,
    sae_dtype=dtype_from_name(SAE_DTYPE),
    top_n=12,
    refresh=REFRESH_FEATURE,
)

print("feature loaded from cache:", loaded_from_cache)
display(handle.as_row())
if discovery_rows:
    display(discovery_rows[:12])


In [ ]:
modes = [
    "remove_activation",
    "projection_remove",
    "subtract_unit",
    "add_activation",
    "add_unit",
]
rows = intervention_mode_rows(
    lm,
    CASE,
    layer=handle.layer,
    sae=sae,
    feature_id=handle.feature_id,
    feature_value=handle.feature_value,
    modes=modes,
    coefficient=1.0,
)
display(rows)


해석 체크: 제거와 projection 제거가 모두 margin을 낮추면 해당 direction 자체가 중요할 가능성이 큽니다. add 계열에서 반대 방향 효과가 나오면 steering 가능성이 커집니다.